# URBANopt Post-Process Setup (Path-Only)

This notebook only initializes `uo_analysis` from a single URBANopt project path.

- No Modelica loading
- No DES result processing
- Intended for cases like:
  `/Users/nlong/working/urban-analysis/esbe/esbe_2026/activity_3/coincident`

In [ ]:
import warnings
from pathlib import Path

from urbanopt_des.urbanopt_analysis import URBANoptAnalysis

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

In [ ]:
# Single input: can be either a URBANopt project dir or a scenario run dir.
# Example outside this repo: /Users/nlong/working/junk
input_path = Path("/Users/nlong/working/junk").expanduser()
# Optional override. If None, choose baseline_scenario when available, else first run folder.
scenario_name = None


if not input_path.exists():
    raise FileNotFoundError(f"Input path does not exist: {input_path}")

year = 2017

# Resolve whether input_path is a project directory or a scenario directory.
if (input_path / "run").exists():
    uo_project_dir = input_path
    run_dir = uo_project_dir / "run"
elif input_path.parent.name == "run":
    uo_project_dir = input_path.parent.parent
    run_dir = uo_project_dir / "run"
    scenario_name = input_path.name
else:
    raise ValueError(
        "Input path must be a URBANopt project dir (contains run/) or a scenario dir under run/."
    )

if not run_dir.exists():
    raise FileNotFoundError(f"Run directory not found: {run_dir}")

if scenario_name is None:
    if (run_dir / "baseline_scenario").exists():
        scenario_name = "baseline_scenario"
    else:
        scenario_dirs = sorted([p for p in run_dir.iterdir() if p.is_dir()])
        if not scenario_dirs:
            raise FileNotFoundError(f"No scenario directories found in: {run_dir}")
        scenario_name = scenario_dirs[0].name

scenario_results_dir = run_dir / scenario_name
if not scenario_results_dir.exists():
    raise FileNotFoundError(f"Scenario directory not found: {scenario_results_dir}")

# Auto-discover a project geojson in the project directory.
geojson_candidates = sorted(uo_project_dir.glob("class_project*.json"))
if not geojson_candidates:
    geojson_candidates = sorted(uo_project_dir.glob("*.json"))
if not geojson_candidates:
    raise FileNotFoundError(f"No project GeoJSON found in: {uo_project_dir}")

project_geojson_filename = geojson_candidates[0]
analysis_dir = uo_project_dir

print(f"Input path: {input_path}")
print(f"URBANopt project dir: {uo_project_dir}")
print(f"Scenario: {scenario_name}")
print(f"GeoJSON: {project_geojson_filename}")
print(f"Results dir: {scenario_results_dir}")

In [ ]:
# Create URBANopt analysis object and load URBANopt results only
uo_analysis = URBANoptAnalysis(project_geojson_filename, analysis_dir, year)

uo_analysis.add_urbanopt_results(uo_project_dir, scenario_name)

# Some buildings may not have load-export files; continue with a warning.
try:
    uo_analysis.urbanopt.process_load_results(uo_analysis.geojson.get_building_ids())
except Exception as exc:
    warnings.warn(
        f"Skipping missing building load exports for now: {exc}", RuntimeWarning
    )

uo_analysis.urbanopt.create_aggregations(uo_analysis.geojson.get_building_ids())
uo_analysis.urbanopt.save_dataframes()
uo_analysis.urbanopt.display_name = "Non-Connected"

geojson_building_ids = uo_analysis.geojson.get_building_ids()

# Keep additional variable list minimal and robust.
other_vars_to_gather = []

uo_analysis.resample_and_convert_modelica_results([], other_vars_to_gather)
uo_analysis.save_modelica_variables()

# Save a copy of URBANopt OpenStudio results in the analysis directory.
uo_analysis.save_urbanopt_results_in_modelica_paths()

# Create combined dataframes between Modelica and OpenStudio.
uo_analysis.combine_modelica_and_openstudio_results()

# Load actual data and create aggregations.
uo_analysis.resample_actual_data()

# Save combined data frames for testing.
uo_analysis.save_dataframes("min_60_with_buildings")

# Aggregations across columns.
uo_analysis.create_modelica_aggregations()

# Roll up to monthly, annual, etc.
uo_analysis.create_rollups()

# Create building summary tables.
uo_analysis.create_building_summaries()

# Save resulting dataframes (including carbon metrics if present).
uo_analysis

In [ ]:
uo_analysis.calculate_all_grid_metrics()
# display the grid metric for one of the analysis
# display(uo_analysis.modelica['controlled'].grid_metrics_annual)

# still need to add in utility costs
# uo_analysis.calculate_utility_costs()

# save the dataframes, grid metrics only
uo_analysis.save_dataframes(["grid_metrics_daily", "grid_metrics_annual"])

# Combine all the results of the analyses for comparison
uo_analysis.create_summary_results()

# print(f"Analysis results are being saved to {uo_analysis.analysis_dir}")
uo_analysis.save_dataframes(["grid_summary", "end_use_summary"])

# create a mapping for the names... but this should go away eventually!
analysis_name_mappings = uo_analysis.display_name_mappings()

# Create building level results for the UO and Modelica results
buildings_df = uo_analysis.create_building_level_results()
buildings_df.to_csv(
    uo_analysis.urbanopt.scenario_output_path / "building_metrics_annual.csv",
    index=True,
)
print(
    f"Saved to {uo_analysis.urbanopt.scenario_output_path / 'building_metrics_annual.csv'}"
)

display(buildings_df)

### Simple Graphs

In [ ]:
# end use names and colors. The names have to match what is in the end_use_summary results dataframe
energy_end_use_rows = {
    "Interior Lighting": "#FFFFCC",
    "Exterior Lighting": "lightblue",
    "Plug Loads": "brown",
    "Building Cooling": "blue",
    "District Cooling": "blue",
    "District Heating": "orange",
    "Building Heating": "orange",
    "Building Fans": "lightgray",
    "Building Pumps": "lightblue",
    "Building Heat Rejection": "royalblue",
    "Building Water Systems": "#FFBB78",
    "ETS Pump Total": "lightgreen",
    "ETS Heat Pump": "gold",
    "Sewer Pump": "darkgray",
    "GHX Pump": "darkgreen",
    "Distribution Pump": "darkblue",
}

In [ ]:
# Plot only the Non-Connected end uses
# drop all columns except "Units" and "Non-Connected"
df_to_plot = uo_analysis.end_use_summary.copy()
columns_to_drop = [col for col in df_to_plot.columns if col not in ["Non-Connected"]]
df_to_plot = df_to_plot.drop(columns=columns_to_drop)

# remove the rows where the "Non-Connected" values are zero
df_to_plot = df_to_plot[df_to_plot["Non-Connected"] > 0]
# copy the energy_end_use_rows and remove the ones that are not in the df_to_plot
end_use_rows_sub = energy_end_use_rows.copy()
rows_with_data = df_to_plot.index.tolist()
end_use_rows_sub = {
    key: value for key, value in end_use_rows_sub.items() if key in rows_with_data
}

df_to_plot = df_to_plot.T / 1e6
df_to_plot = df_to_plot[end_use_rows_sub.keys()]

# save off the data for sending
df_to_plot.to_csv(results_summary_dir / "end_use_summary_non_connected.csv")
print(f"Saved plot to {results_summary_dir / 'end_use_summary_non_connected.csv'}")

# plot the end use summary with the updated colors
df_to_plot.plot.bar(
    figsize=graph_size, stacked=True, color=end_use_rows_sub.values()
)  # title=f"Analysis Summary"
plt.legend(
    fontsize="x-small", bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0
)
plt.tight_layout()
plt.subplots_adjust(left=0.1)  # fix the left axis to show the full text
plt.ylabel("Total Energy [MWh]")
# rotate labels on xaxis
plt.xticks(rotation=0)
# add thousands separator to y-axis
plt.gca().get_yaxis().set_major_formatter(
    plt.FuncFormatter(lambda x, _loc: f"{int(x):,}")
)

output_name = results_summary_dir / "bar_end_use_summary_all.png"
plt.savefig(output_name, dpi=300)
plt.show()